# Notebook 3: Chronos-2 offline comparison

Project 4, retail demand forecasting on FreshRetailNet-50K. Runs on its own: downloads the same
pinned dataset and writes the same evaluation module as notebooks 1 and 2.

Chronos-2 (Amazon, Apache 2.0, 120M parameters) is a pretrained time-series model that forecasts
without training on this dataset. This notebook answers one question with measurements rather than
argument: **is it worth serving instead of the LightGBM model?** That means two numbers, accuracy
on the same evaluation week and inference throughput, not just the first.

The comparison is deliberately offline. The served model stays LightGBM unless Chronos-2 beats it
on both.

Kaggle settings: Internet on. GPU recommended (set Accelerator to a GPU); it runs on CPU, slower.

## 1. Configuration

In [1]:
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa

DATASET_REPO = "Dingdong-Inc/FreshRetailNet-50K"
DATASET_REVISION = "08c1fab7f9257bc73679d415d65d644165d351d4"
DATASET_FILES = {
    "train": ("data/train.parquet", "6706832db892bbae4969c19d87e07975d2543d2ba7d7d4756360654785de5a3d"),
    "eval": ("data/eval.parquet", "1b118840664280c6b88bffc84c80ee1f54c05d911e354b7599e5da10995e960e"),
}
CACHE_DIR = Path.home() / ".cache" / "freshretailnet-50k" / DATASET_REVISION
KAGGLE_WORKING = Path("/kaggle/working")
OUTPUT_DIR = KAGGLE_WORKING if KAGGLE_WORKING.exists() else Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODULE_DIR = Path.cwd()
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

HORIZON_DAYS = 7
SEED = 20240626
CONTEXT_DAYS = int(os.environ.get("CONTEXT_DAYS", 90))
SUBSET_SERIES = int(os.environ.get("SUBSET_SERIES", 0))          # 0 uses all 50,000 series
BATCH_SIZE = int(os.environ.get("CHRONOS_BATCH_SIZE", 256))
QUANTILES = [0.1, 0.5, 0.9]
THROUGHPUT_SERIES = int(os.environ.get("THROUGHPUT_SERIES", 512))
CPU_BENCHMARK_SERIES = int(os.environ.get("CPU_BENCHMARK_SERIES", 128))

# measured in notebook 2 on the same eval week, quoted here for comparison
LIGHTGBM_EVAL = {"wape": 0.33168907433084976, "wpe": -0.03410891931355602,
                 "mae": 0.3957166918475688, "rmse": 0.6730458170158612}
# measured on the kind deployment, 10 concurrent users, batching on
SERVICE_MEASURED = {"throughput_rps": 82.2, "p50_ms": 100.0, "p99_ms": 390.0,
                    "model_predict_ms_per_request": 5.4}

report = {"checks": {}}


def check(name, condition, detail=None):
    report["checks"][name] = {"passed": bool(condition), "detail": detail}
    print(("PASS " if condition else "FAIL ") + name + ("" if detail is None else f" | {detail}"))
    assert condition, name

In [2]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "chronos-forecasting>=2.0"], check=True)

import torch
from chronos import Chronos2Pipeline

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
environment = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "pyarrow": pa.__version__,
    "torch": torch.__version__,
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
    "on_kaggle": KAGGLE_WORKING.exists(),
    "context_days": CONTEXT_DAYS,
    "subset_series": SUBSET_SERIES,
}
print(json.dumps(environment, indent=2))

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "python": "3.11.15",
  "platform": "Linux-6.18.44-fc-v33-x86_64-with-glibc2.39",
  "numpy": "2.4.4",
  "pandas": "3.0.2",
  "pyarrow": "25.0.1",
  "torch": "2.14.0+cu130",
  "device": "cpu",
  "gpu": null,
  "on_kaggle": false,
  "context_days": 90,
  "subset_series": 300
}


## 2. Download at the pinned revision

In [3]:
def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def fetch(split):
    rel, expected = DATASET_FILES[split]
    dest = CACHE_DIR / rel
    if not (dest.exists() and sha256_of(dest) == expected):
        dest.parent.mkdir(parents=True, exist_ok=True)
        url = f"https://huggingface.co/datasets/{DATASET_REPO}/resolve/{DATASET_REVISION}/{rel}"
        tmp = dest.with_suffix(".part")
        with urllib.request.urlopen(url, timeout=120) as resp, open(tmp, "wb") as out:
            shutil.copyfileobj(resp, out, 1 << 20)
        tmp.replace(dest)
    assert sha256_of(dest) == expected, f"{split}: sha256 mismatch"
    print(f"{split}: {dest.stat().st_size:,} bytes, sha256 ok")
    return dest


TRAIN_PATH, EVAL_PATH = fetch("train"), fetch("eval")

train: 106,436,287 bytes, sha256 ok


eval: 8,440,124 bytes, sha256 ok


## 3. Evaluation module

In [4]:
%%writefile demand_features.py
"""Leakage-safe daily demand features for FreshRetailNet-50K.

Every history-derived feature for target day t reads only days <= t - MIN_LAG.
With MIN_LAG equal to the forecast horizon, one feature definition serves every
horizon step 1..HORIZON from a single forecast origin (direct strategy), and the
same code path is used for training rows and for serving requests.

Same-day columns are split into two groups:
- KNOWN_FUTURE_COLS: planned in advance (calendar, discount, promo activity).
  Treated as known for the target day. Discount is an assumption, see notebook 1.
- Realized same-day columns (target, stockout hours, weather): never used for
  the target day. Weather is only available through the explicit "oracle" mode,
  which exists for a labelled sensitivity comparison and not for serving.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

KEY_COLS = ["store_id", "product_id"]
DATE_COL = "dt"
TARGET_COL = "sale_amount"
STOCK_COL = "stock_hour6_22_cnt"
STATIC_COLS = [
    "city_id",
    "store_id",
    "management_group_id",
    "first_category_id",
    "second_category_id",
    "third_category_id",
    "product_id",
]
KNOWN_FUTURE_COLS = ["discount", "activity_flag", "holiday_flag"]
WEATHER_COLS = ["precpt", "avg_temperature", "avg_humidity", "avg_wind_level"]
REALIZED_COLS = [TARGET_COL, STOCK_COL] + WEATHER_COLS

HORIZON = 7
MIN_LAG = HORIZON
SALES_LAGS = (7, 8, 9, 10, 11, 12, 13, 14, 21, 28)
SALES_MEAN_WINDOWS = (7, 14, 28)
SALES_STD_WINDOWS = (7, 28)
STOCK_MEAN_WINDOWS = (7, 28)
FULL_DAY_STOCKOUT_HOURS = 16
WEATHER_MODES = ("none", "oracle")

REQUIRED_HISTORY_DAYS = max(max(SALES_LAGS), MIN_LAG + max(SALES_MEAN_WINDOWS) - 1)


class PanelError(ValueError):
    pass


def to_panel(df: pd.DataFrame, value_cols: list[str]) -> tuple[pd.DataFrame, pd.DatetimeIndex, dict[str, np.ndarray]]:
    """Reshape a long frame into (n_series, n_days) arrays.

    Requires a complete, duplicate-free panel over a contiguous daily range.
    """
    missing = [c for c in KEY_COLS + [DATE_COL] + value_cols if c not in df.columns]
    if missing:
        raise PanelError(f"missing columns: {missing}")
    dates = pd.to_datetime(df[DATE_COL])
    start, end = dates.min(), dates.max()
    grid = pd.date_range(start, end, freq="D")
    n_days = len(grid)
    keys = df[KEY_COLS].drop_duplicates().sort_values(KEY_COLS).reset_index(drop=True)
    n_series = len(keys)
    if len(df) != n_series * n_days:
        raise PanelError(
            f"incomplete or duplicated panel: {len(df)} rows, expected {n_series} x {n_days} = {n_series * n_days}"
        )
    order = np.lexsort((dates.to_numpy(), df["product_id"].to_numpy(), df["store_id"].to_numpy()))
    sorted_dates = dates.to_numpy()[order].reshape(n_series, n_days)
    if not (sorted_dates == grid.to_numpy()[None, :]).all():
        raise PanelError("panel dates are not a complete contiguous grid for every series")
    sorted_keys = df[KEY_COLS].to_numpy()[order].reshape(n_series, n_days, len(KEY_COLS))
    if not (sorted_keys == sorted_keys[:, :1, :]).all():
        raise PanelError("series keys are not constant along the date axis after sorting")
    arrays = {c: df[c].to_numpy(dtype=np.float64)[order].reshape(n_series, n_days) for c in value_cols}
    return keys, grid, arrays


def lag(x: np.ndarray, k: int) -> np.ndarray:
    out = np.full_like(x, np.nan, dtype=np.float64)
    if k < x.shape[1]:
        out[:, k:] = x[:, : x.shape[1] - k]
    return out


def _window_sums(x: np.ndarray, window: int, offset: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Sums over days (t - offset - window + 1 .. t - offset), accumulated window element by element.

    Each output value depends only on the values inside its own window, so the result is bit-identical
    regardless of how much history precedes the window (required for serving parity).
    """
    s1 = np.zeros(x.shape, dtype=np.float64)
    s2 = np.zeros(x.shape, dtype=np.float64)
    n = np.zeros(x.shape, dtype=np.int32)
    for j in range(window):
        v = lag(x, offset + j)
        valid = ~np.isnan(v)
        v0 = np.where(valid, v, 0.0)
        s1 += v0
        s2 += v0 * v0
        n += valid
    return s1, s2, n


def rolling_mean(x: np.ndarray, window: int, offset: int) -> np.ndarray:
    """Mean over days (t - offset - window + 1 .. t - offset). NaN unless the full window is observed."""
    s1, _, n = _window_sums(x, window, offset)
    return np.where(n == window, s1 / window, np.nan)


def rolling_std(x: np.ndarray, window: int, offset: int) -> np.ndarray:
    """Population standard deviation over the same window as rolling_mean."""
    s1, s2, n = _window_sums(x, window, offset)
    mean = s1 / window
    var = np.maximum(s2 / window - mean * mean, 0.0)
    return np.where(n == window, np.sqrt(var), np.nan)


def feature_columns(weather_mode: str = "none") -> list[str]:
    if weather_mode not in WEATHER_MODES:
        raise ValueError(f"weather_mode must be one of {WEATHER_MODES}")
    cols = list(STATIC_COLS) + ["day_of_week"] + list(KNOWN_FUTURE_COLS)
    cols += [f"sales_lag_{k}" for k in SALES_LAGS]
    cols += [f"sales_mean_{w}_off{MIN_LAG}" for w in SALES_MEAN_WINDOWS]
    cols += [f"sales_std_{w}_off{MIN_LAG}" for w in SALES_STD_WINDOWS]
    cols += ["sales_same_dow_mean_4w", f"sales_zero_share_28_off{MIN_LAG}"]
    cols += [f"stock_hours_lag_{MIN_LAG}"]
    cols += [f"stock_hours_mean_{w}_off{MIN_LAG}" for w in STOCK_MEAN_WINDOWS]
    cols += [f"stock_fullday_share_28_off{MIN_LAG}"]
    cols += [f"discount_mean_7_off{MIN_LAG}", "discount_vs_recent", f"activity_share_28_off{MIN_LAG}"]
    if weather_mode == "oracle":
        cols += [f"oracle_{c}" for c in WEATHER_COLS]
    return cols


def build_feature_arrays(
    arrays: dict[str, np.ndarray],
    grid: pd.DatetimeIndex,
    weather_mode: str = "none",
) -> dict[str, np.ndarray]:
    """History and known-future features as (n_series, n_days) arrays. Static columns excluded."""
    if weather_mode not in WEATHER_MODES:
        raise ValueError(f"weather_mode must be one of {WEATHER_MODES}")
    y = arrays[TARGET_COL]
    stock = arrays[STOCK_COL]
    n_series, n_days = y.shape
    f: dict[str, np.ndarray] = {}

    f["day_of_week"] = np.broadcast_to(grid.dayofweek.to_numpy(dtype=np.float64)[None, :], (n_series, n_days))
    for c in KNOWN_FUTURE_COLS:
        f[c] = arrays[c]

    for k in SALES_LAGS:
        f[f"sales_lag_{k}"] = lag(y, k)
    for w in SALES_MEAN_WINDOWS:
        f[f"sales_mean_{w}_off{MIN_LAG}"] = rolling_mean(y, w, MIN_LAG)
    for w in SALES_STD_WINDOWS:
        f[f"sales_std_{w}_off{MIN_LAG}"] = rolling_std(y, w, MIN_LAG)
    f["sales_same_dow_mean_4w"] = (lag(y, 7) + lag(y, 14) + lag(y, 21) + lag(y, 28)) / 4.0
    zero = np.where(np.isnan(y), np.nan, (y == 0).astype(np.float64))
    f[f"sales_zero_share_28_off{MIN_LAG}"] = rolling_mean(zero, 28, MIN_LAG)

    f[f"stock_hours_lag_{MIN_LAG}"] = lag(stock, MIN_LAG)
    for w in STOCK_MEAN_WINDOWS:
        f[f"stock_hours_mean_{w}_off{MIN_LAG}"] = rolling_mean(stock, w, MIN_LAG)
    fullday = np.where(np.isnan(stock), np.nan, (stock >= FULL_DAY_STOCKOUT_HOURS).astype(np.float64))
    f[f"stock_fullday_share_28_off{MIN_LAG}"] = rolling_mean(fullday, 28, MIN_LAG)

    disc_recent = rolling_mean(arrays["discount"], 7, MIN_LAG)
    f[f"discount_mean_7_off{MIN_LAG}"] = disc_recent
    f["discount_vs_recent"] = arrays["discount"] - disc_recent
    f[f"activity_share_28_off{MIN_LAG}"] = rolling_mean(arrays["activity_flag"], 28, MIN_LAG)

    if weather_mode == "oracle":
        for c in WEATHER_COLS:
            f[f"oracle_{c}"] = arrays[c]
    return f


def mask_realized(arrays: dict[str, np.ndarray], grid: pd.DatetimeIndex, first_masked_date) -> dict[str, np.ndarray]:
    """Copy of arrays with realized columns set to NaN from first_masked_date onward."""
    idx = int(np.searchsorted(grid.to_numpy(), np.datetime64(pd.Timestamp(first_masked_date))))
    out = dict(arrays)
    for c in REALIZED_COLS:
        if c in out:
            a = out[c].copy()
            a[:, idx:] = np.nan
            out[c] = a
    return out


def panel_value_cols(weather_mode: str = "none") -> list[str]:
    cols = [TARGET_COL, STOCK_COL] + list(KNOWN_FUTURE_COLS)
    if weather_mode == "oracle":
        cols += list(WEATHER_COLS)
    return cols


def build_features(
    df: pd.DataFrame,
    weather_mode: str = "none",
    first_masked_date=None,
    rows_from_date=None,
) -> pd.DataFrame:
    """Long feature frame: keys, dt, feature_columns(weather_mode).

    first_masked_date: realized columns from this date on are removed before any
    feature is computed. rows_from_date: only rows on or after this date are returned.
    """
    keys, grid, arrays = to_panel(df, panel_value_cols(weather_mode))
    if first_masked_date is not None:
        arrays = mask_realized(arrays, grid, first_masked_date)
    feats = build_feature_arrays(arrays, grid, weather_mode)

    static = df.drop_duplicates(KEY_COLS)[STATIC_COLS].copy()
    static = keys.merge(static, on=KEY_COLS, how="left", validate="one_to_one")
    if static[STATIC_COLS].isna().any().any():
        raise PanelError("static attributes missing for some series")

    start = 0
    if rows_from_date is not None:
        start = int(np.searchsorted(grid.to_numpy(), np.datetime64(pd.Timestamp(rows_from_date))))
    n_series = len(keys)
    days = grid[start:]
    n_days = len(days)

    out = {}
    for c in STATIC_COLS:
        out[c] = np.repeat(static[c].to_numpy(), n_days)
    out[DATE_COL] = np.tile(days.to_numpy(), n_series)
    for name in feature_columns(weather_mode):
        if name in STATIC_COLS:
            continue
        out[name] = np.ascontiguousarray(feats[name][:, start:]).reshape(-1).astype(np.float32)
    frame = pd.DataFrame(out)
    return frame[STATIC_COLS + [DATE_COL] + [c for c in feature_columns(weather_mode) if c not in STATIC_COLS]]

Writing demand_features.py


In [5]:
%%writefile demand_eval.py
"""Baselines and metrics for fixed-origin 7-day forecasts.

All baselines receive the target panel with every day after the forecast origin
set to NaN, and fail if any prediction is non-finite, so a baseline that reads
past the origin cannot silently produce numbers.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from demand_features import HORIZON

BASELINES = ("seasonal_naive_7", "moving_average_7", "same_dow_mean_4w")


def _masked(y: np.ndarray, origin_idx: int) -> np.ndarray:
    m = y.astype(np.float64, copy=True)
    m[:, origin_idx + 1 :] = np.nan
    return m


def baseline_forecast(y: np.ndarray, origin_idx: int, name: str, horizon: int = HORIZON) -> np.ndarray:
    """Forecast of shape (n_series, horizon) for days origin_idx+1 .. origin_idx+horizon."""
    if origin_idx + 1 < 28:
        raise ValueError("origin needs at least 28 observed days")
    hist = _masked(y, origin_idx)
    targets = origin_idx + 1 + np.arange(horizon)
    if name == "seasonal_naive_7":
        pred = hist[:, targets - 7]
    elif name == "moving_average_7":
        pred = np.repeat(hist[:, origin_idx - 6 : origin_idx + 1].mean(axis=1, keepdims=True), horizon, axis=1)
    elif name == "same_dow_mean_4w":
        pred = np.mean(np.stack([hist[:, targets - k] for k in (7, 14, 21, 28)]), axis=0)
    else:
        raise ValueError(f"unknown baseline {name}")
    if not np.isfinite(pred).all():
        raise RuntimeError(f"{name} produced non-finite values; it read data after the origin")
    return pred


def point_metrics(actual, pred) -> dict[str, float]:
    a = np.asarray(actual, dtype=np.float64)
    p = np.asarray(pred, dtype=np.float64)
    err = p - a
    total = a.sum()
    return {
        "n": int(a.size),
        "sum_actual": float(total),
        "wape": float(np.abs(err).sum() / total) if total > 0 else float("nan"),
        "wpe": float(err.sum() / total) if total > 0 else float("nan"),
        "mae": float(np.abs(err).mean()),
        "rmse": float(np.sqrt((err * err).mean())),
    }


def metrics_by(frame: pd.DataFrame, by: str, actual: str = "actual", pred: str = "pred") -> pd.DataFrame:
    rows = []
    for key, g in frame.groupby(by, observed=True, sort=True):
        rows.append({by: key, **point_metrics(g[actual].to_numpy(), g[pred].to_numpy())})
    return pd.DataFrame(rows)


def stockout_bucket(hours) -> pd.Categorical:
    h = np.asarray(hours)
    labels = np.where(h == 0, "0h", np.where(h >= 16, "16h_full_day", "1-15h"))
    return pd.Categorical(labels, categories=["0h", "1-15h", "16h_full_day"], ordered=True)

Writing demand_eval.py


In [6]:
import demand_eval as de
import demand_features as dfe

## 4. Inputs for Chronos-2

Chronos-2 reads a long frame of context and, optionally, a frame of future values. Columns present
in both frames are treated as known in advance; columns only in the context frame are past-only
covariates. The split matches what the LightGBM model is allowed to see:

- past only: `stock_hour6_22_cnt`, `discount`
- known in advance: `activity_flag`, `holiday_flag`
- target: `sale_amount`

No same-day discount, no weather, exactly as in the served configuration.

In [7]:
load_cols = list(dict.fromkeys(dfe.KEY_COLS + [dfe.DATE_COL] + dfe.panel_value_cols("none")))
train = pd.read_parquet(TRAIN_PATH, columns=load_cols)
evald = pd.read_parquet(EVAL_PATH, columns=list(dict.fromkeys(load_cols + [dfe.STOCK_COL])))
for frame in (train, evald):
    frame[dfe.DATE_COL] = pd.to_datetime(frame[dfe.DATE_COL])

keys = train[dfe.KEY_COLS].drop_duplicates().sort_values(dfe.KEY_COLS).reset_index(drop=True)
if SUBSET_SERIES:
    keys = keys.sample(SUBSET_SERIES, random_state=SEED).sort_values(dfe.KEY_COLS).reset_index(drop=True)
    train = train.merge(keys, on=dfe.KEY_COLS, how="inner")
    evald = evald.merge(keys, on=dfe.KEY_COLS, how="inner")
    print(f"subset: {len(keys):,} series")

train = train.sort_values(dfe.KEY_COLS + [dfe.DATE_COL]).reset_index(drop=True)
evald = evald.sort_values(dfe.KEY_COLS + [dfe.DATE_COL]).reset_index(drop=True)
train_dates = pd.DatetimeIndex(sorted(train[dfe.DATE_COL].unique()))
eval_dates = pd.DatetimeIndex(sorted(evald[dfe.DATE_COL].unique()))
context_start = train_dates[-CONTEXT_DAYS]
check("eval window is the 7 days after the training window",
      len(eval_dates) == HORIZON_DAYS and eval_dates[0] == train_dates[-1] + pd.Timedelta(days=1),
      f"{eval_dates[0].date()}..{eval_dates[-1].date()}")

series_id = (train["store_id"].astype(str) + "_" + train["product_id"].astype(str))
context_df = pd.DataFrame({
    "item_id": series_id,
    "timestamp": train[dfe.DATE_COL],
    "target": train[dfe.TARGET_COL].astype("float32"),
    "stock_hours": train[dfe.STOCK_COL].astype("float32"),
    "discount": train["discount"].astype("float32"),
    "activity_flag": train["activity_flag"].astype("float32"),
    "holiday_flag": train["holiday_flag"].astype("float32"),
})
context_df = context_df[train[dfe.DATE_COL] >= context_start].reset_index(drop=True)
future_df = pd.DataFrame({
    "item_id": evald["store_id"].astype(str) + "_" + evald["product_id"].astype(str),
    "timestamp": evald[dfe.DATE_COL],
    "activity_flag": evald["activity_flag"].astype("float32"),
    "holiday_flag": evald["holiday_flag"].astype("float32"),
})
check("context and future frames cover the same series",
      context_df["item_id"].nunique() == future_df["item_id"].nunique() == len(keys),
      f"{len(keys):,} series, context {len(context_df):,} rows, future {len(future_df):,} rows")
check("context holds the expected number of days per series",
      int(context_df.groupby("item_id").size().max()) == CONTEXT_DAYS, CONTEXT_DAYS)

subset: 300 series
PASS eval window is the 7 days after the training window | 2024-06-26..2024-07-02
PASS context and future frames cover the same series | 300 series, context 27,000 rows, future 2,100 rows
PASS context holds the expected number of days per series | 90


## 5. Zero-shot forecasts

In [8]:
pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map=DEVICE)
t0 = time.perf_counter()
forecast = pipeline.predict_df(
    context_df,
    future_df=future_df,
    prediction_length=HORIZON_DAYS,
    quantile_levels=QUANTILES,
    batch_size=BATCH_SIZE,
    id_column="item_id",
    timestamp_column="timestamp",
    target="target",
)
elapsed = time.perf_counter() - t0
print(f"{len(keys):,} series in {elapsed:.1f}s on {DEVICE} "
      f"({len(keys)/elapsed:.1f} series/s, {len(keys)*HORIZON_DAYS/elapsed:,.0f} forecast rows/s)")

forecast = forecast.rename(columns={"0.5": "chronos_p50", "0.1": "chronos_p10", "0.9": "chronos_p90"})
forecast["chronos"] = np.clip(forecast["chronos_p50"].to_numpy(), 0.0, None)
evald["item_id"] = evald["store_id"].astype(str) + "_" + evald["product_id"].astype(str)
merged = evald.merge(forecast[["item_id", "timestamp", "chronos", "chronos_p10", "chronos_p90"]],
                     left_on=["item_id", dfe.DATE_COL], right_on=["item_id", "timestamp"], how="left")
check("a forecast exists for every eval row", int(merged["chronos"].isna().sum()) == 0, f"{len(merged):,} rows")

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 170/170 [00:00<00:00, 8062.69it/s]

300 series in 13.5s on cpu (22.2 series/s, 155 forecast rows/s)
PASS a forecast exists for every eval row | 2,100 rows


## 6. Accuracy on the same evaluation week

The baselines are recomputed here from the same data, so this notebook does not depend on notebook
2's outputs. The LightGBM figure is quoted from notebook 2's model card, measured on exactly these
rows. If notebook 2's predictions file is available, set `NB2_PREDICTIONS` and the comparison
becomes row-aligned instead of quoted.

In [9]:
keys_arr, grid, panel = dfe.to_panel(
    pd.concat([train, evald[train.columns]], ignore_index=True), [dfe.TARGET_COL])
origin_idx = len(grid) - HORIZON_DAYS - 1
baseline_pred = {name: de.baseline_forecast(panel[dfe.TARGET_COL], origin_idx, name).reshape(-1)
                 for name in de.BASELINES}
order = (keys_arr["store_id"].astype(str) + "_" + keys_arr["product_id"].astype(str))
baseline_frame = pd.DataFrame({
    "item_id": np.repeat(order.to_numpy(), HORIZON_DAYS),
    dfe.DATE_COL: np.tile(grid[-HORIZON_DAYS:].to_numpy(), len(keys_arr)),
    **{f"pred_{k}": v for k, v in baseline_pred.items()},
})
merged = merged.merge(baseline_frame, on=["item_id", dfe.DATE_COL], how="left", validate="one_to_one")

results = {"chronos2_zeroshot": de.point_metrics(merged[dfe.TARGET_COL], merged["chronos"])}
for name in de.BASELINES:
    results[name] = de.point_metrics(merged[dfe.TARGET_COL], merged[f"pred_{name}"])
results["lightgbm_served_notebook2"] = dict(LIGHTGBM_EVAL, n=len(merged), sum_actual=float(merged[dfe.TARGET_COL].sum()))

nb2_path = os.environ.get("NB2_PREDICTIONS", "")
if nb2_path and Path(nb2_path).exists():
    nb2 = pd.read_parquet(nb2_path)
    nb2["item_id"] = nb2["store_id"].astype(str) + "_" + nb2["product_id"].astype(str)
    nb2[dfe.DATE_COL] = pd.to_datetime(nb2["dt"])
    col = [c for c in nb2.columns if c.startswith("pred_lightgbm") and "no_same_day_discount" in c][0]
    merged = merged.merge(nb2[["item_id", dfe.DATE_COL, col]], on=["item_id", dfe.DATE_COL], how="left")
    merged = merged.rename(columns={col: "lightgbm"})
    results["lightgbm_served_rowaligned"] = de.point_metrics(merged[dfe.TARGET_COL], merged["lightgbm"])
    print("row-aligned LightGBM comparison enabled")

table = pd.DataFrame(results).T[["wape", "wpe", "mae", "rmse"]]
print(table.round(4).sort_values("wape").to_string())

                             wape     wpe     mae    rmse
lightgbm_served_notebook2  0.3317 -0.0341  0.3957  0.6730
chronos2_zeroshot          0.3624 -0.0682  0.4166  0.6694
moving_average_7           0.3689 -0.0071  0.4241  0.6647
same_dow_mean_4w           0.3998 -0.0860  0.4596  0.7281
seasonal_naive_7           0.4221 -0.0071  0.4853  0.7366


In [10]:
merged["horizon_day"] = merged.groupby("item_id").cumcount() + 1
merged["stockout_bucket"] = de.stockout_bucket(merged[dfe.STOCK_COL])
breakdowns = {}
for label, column in (("chronos2_zeroshot", "chronos"), ("moving_average_7", "pred_moving_average_7")):
    frame = merged.rename(columns={dfe.TARGET_COL: "actual", column: "pred"})
    breakdowns[label] = {
        part: de.metrics_by(frame, part).astype({part: str}).to_dict(orient="records")
        for part in ("horizon_day", "stockout_bucket")
    }
    for part in ("horizon_day", "stockout_bucket"):
        print(f"\n{label} by {part}")
        print(pd.DataFrame(breakdowns[label][part]).drop(columns=["sum_actual"]).round(4).to_string(index=False))

coverage = float(((merged[dfe.TARGET_COL] >= merged["chronos_p10"]) &
                  (merged[dfe.TARGET_COL] <= merged["chronos_p90"])).mean())
print(f"\nChronos-2 80% interval coverage: {coverage:.3f} (nominal 0.8)")


chronos2_zeroshot by horizon_day
horizon_day   n   wape     wpe    mae   rmse
          1 300 0.3831  0.0142 0.3804 0.6189
          2 300 0.3500 -0.0573 0.3694 0.5662
          3 300 0.3449 -0.0086 0.3619 0.5529
          4 300 0.3258 -0.1057 0.4375 0.6617
          5 300 0.3816 -0.2074 0.5704 0.9094
          6 300 0.3774  0.0366 0.3844 0.5932
          7 300 0.3770 -0.0722 0.4124 0.7145

chronos2_zeroshot by stockout_bucket
stockout_bucket    n    wape     wpe    mae   rmse
             0h 1270  0.3366 -0.0825 0.3910 0.6458
          1-15h  767  0.3489 -0.1008 0.4261 0.6645
   16h_full_day   63 21.4779 21.3931 0.8182 1.0740

moving_average_7 by horizon_day
horizon_day   n   wape     wpe    mae   rmse
          1 300 0.3986  0.1496 0.3958 0.6103
          2 300 0.3523  0.0815 0.3718 0.5675
          3 300 0.3457  0.0880 0.3627 0.5403
          4 300 0.3273 -0.1498 0.4394 0.6766
          5 300 0.3906 -0.2362 0.5837 0.9142
          6 300 0.3789  0.1207 0.3859 0.5844
          7 300 

## 7. Inference cost

Accuracy is only half the serving question. This measures Chronos-2's throughput on the device it
is running on, and on CPU, against the LightGBM service's measured figures from the kind
deployment.

In [11]:
bench_ids = context_df["item_id"].drop_duplicates().head(THROUGHPUT_SERIES)
bench_ctx = context_df[context_df["item_id"].isin(bench_ids)]
bench_fut = future_df[future_df["item_id"].isin(bench_ids)]


def benchmark(pipe, ctx, fut, n_series, repeats=2):
    timings = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        pipe.predict_df(ctx, future_df=fut, prediction_length=HORIZON_DAYS, quantile_levels=QUANTILES,
                        batch_size=BATCH_SIZE, id_column="item_id", timestamp_column="timestamp", target="target")
        timings.append(time.perf_counter() - t0)
    best = min(timings)
    return {"series": int(n_series), "seconds": round(best, 2),
            "series_per_second": round(n_series / best, 1),
            "ms_per_series": round(best / n_series * 1000, 2)}


throughput = {"device": DEVICE, DEVICE: benchmark(pipeline, bench_ctx, bench_fut, len(bench_ids))}
print(f"{DEVICE}: {throughput[DEVICE]}")

if DEVICE == "cuda":
    cpu_ids = bench_ids.head(CPU_BENCHMARK_SERIES)
    cpu_pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cpu")
    throughput["cpu"] = benchmark(cpu_pipeline, bench_ctx[bench_ctx["item_id"].isin(cpu_ids)],
                                  bench_fut[bench_fut["item_id"].isin(cpu_ids)], len(cpu_ids), repeats=1)
    print(f"cpu: {throughput['cpu']}")
    del cpu_pipeline

throughput["lightgbm_service_measured"] = SERVICE_MEASURED
print(json.dumps(throughput, indent=2))

cpu: {'series': 64, 'seconds': 2.83, 'series_per_second': 22.6, 'ms_per_series': 44.27}
{
  "device": "cpu",
  "cpu": {
    "series": 64,
    "seconds": 2.83,
    "series_per_second": 22.6,
    "ms_per_series": 44.27
  },
  "lightgbm_service_measured": {
    "throughput_rps": 82.2,
    "p50_ms": 100.0,
    "p99_ms": 390.0,
    "model_predict_ms_per_request": 5.4
  }
}


## 8. Reading the result

Two cautions belong with these numbers.

**Contamination cannot be ruled out.** Chronos-2's training corpus is not fully disclosed, and this
dataset was published before the model. If the dataset, or series correlated with it, appeared in
pretraining, a zero-shot score here flatters the model. Published work on time-series foundation
models documents exactly this problem, so the comparison is reported as indicative, not decisive.

**Serving is a separate decision from accuracy.** The LightGBM service answers at 82 requests per
second with a p50 of 100 ms on one CPU pod, with the model itself costing about 5 ms per request.
The throughput figures above are what a foundation model would have to match to replace it, on
hardware that costs more.

## 9. Save artifacts

In [12]:
environment["chronos_forecasting"] = subprocess.run(
    [sys.executable, "-m", "pip", "show", "chronos-forecasting"], capture_output=True, text=True
).stdout.split("Version:")[1].split()[0]

out_cols = dfe.KEY_COLS + [dfe.DATE_COL, dfe.TARGET_COL, dfe.STOCK_COL, "horizon_day",
                           "chronos", "chronos_p10", "chronos_p90"] + [f"pred_{n}" for n in de.BASELINES]
merged[out_cols].to_parquet(OUTPUT_DIR / "nb3_chronos_predictions.parquet", index=False)
(OUTPUT_DIR / "nb3_chronos_metrics.json").write_text(json.dumps({
    "eval_window": [str(eval_dates[0].date()), str(eval_dates[-1].date())],
    "series": int(len(keys)), "context_days": CONTEXT_DAYS,
    "overall": results, "breakdowns": breakdowns,
    "interval_coverage_80": coverage,
    "lightgbm_reference": LIGHTGBM_EVAL,
}, indent=2, default=str))
(OUTPUT_DIR / "nb3_throughput.json").write_text(json.dumps(throughput, indent=2))
(OUTPUT_DIR / "nb3_environment.json").write_text(json.dumps(environment, indent=2))
(OUTPUT_DIR / "nb3_checks.json").write_text(json.dumps(report, indent=2, default=str))
for p in sorted(OUTPUT_DIR.glob("nb3_*")):
    print(f"{p.name}: {p.stat().st_size:,} bytes")
failed = [k for k, v in report["checks"].items() if not v["passed"]]
print(f"checks: {len(report['checks'])} run, {len(failed)} failed")

nb3_checks.json: 520 bytes
nb3_chronos_metrics.json: 6,806 bytes
nb3_chronos_predictions.parquet: 59,101 bytes
nb3_environment.json: 312 bytes
nb3_throughput.json: 282 bytes
checks: 4 run, 0 failed
